# Knowledge tests


In [13]:
import sys
sys.path.append("..")

## M3 - B+ implementation

### Simple work

In [ ]:
papel_em_branco = bytearray(4096)

papel_em_branco[0] = 1 #significa que é folha

papel_em_branco[1] = 0 
papel_em_branco[2] = 0 

print("envelope criado e configurado")

In [ ]:
print("O nó é folha?", papel_em_branco[0] == 1)
print("Quantas chaves tem?", papel_em_branco[1] * 256 + papel_em_branco[2])

In [ ]:
chave_bytes = (42).to_bytes(4, byteorder='big')
papel_em_branco[7:11] = chave_bytes

ptr_bytes = (100).to_bytes(4, byteorder='big')
papel_em_branco[11:15] = ptr_bytes

# atualiza a quantidade de chaves

papel_em_branco[1] = 0
papel_em_branco[2] = 1

print("Chave e ponteiro gravados")



In [ ]:
chave_lida = int.from_bytes(papel_em_branco[7:11], byteorder='big')
print("Chave lida:", chave_lida)

ptr_lido = int.from_bytes(papel_em_branco[11:15], byteorder='big')
print("Ponteiro lido:", ptr_lido)

### Node class simple test

In [ ]:
from src.BplusTree import Node

In [ ]:
pagina = bytearray(4096)

no = Node(pagina)

#definindo que é folha
no.setIsLeaf(True)
print("Esse nó é nó folha?", no.isLeaf())

#definindo quantas chaves tem no nó
no.setNumKeys(2)
print("Quantas chaves há nesse nó?", no.getNumKeys())

#definindo qual o valor das chaves no nó e seus ponteiros

no.setKey(0, 42)
no.setPointer(0, 100)
no.setKey(1, 10)
no.setPointer(2, 200)

print(f"O ponteiro {no.getPointer(0)} aponta para a chave {no.getKey(0)}")
print(f"O ponteiro {no.getPointer(1)} aponta para a chave {no.getKey(1)}")



### Leaf Split test

In [14]:
from src.BplusTree import Node, BplusTree

# Valor temporário para representar ausência de próxima folha
NO_NEXT = 0xFFFFFFFF

In [15]:
def get_entries(node):
    """Retorna as chaves e ponteiros válidos de uma folha."""
    entries = []

    for i in range(node.getNumKeys()):
        entries.append((node.getKey(i), node.getPointer(i)))

    return entries

In [16]:
nodes = {}
tree = BplusTree(nodes, order=5)

# Corrige temporariamente o ponteiro inicial da raiz
tree.root.setChildPointer(0, NO_NEXT)

# Preenche a folha até sua capacidade máxima
tree.insert(20, 200)
tree.insert(10, 100)
tree.insert(40, 400)
tree.insert(30, 300)

# Confere se ainda existe apenas uma folha
assert tree.root.getNumKeys() == 4
assert len(nodes) == 1

# Esta inserção deve provocar o split
tree.insert(50, 500)

old_leaf = nodes[0]
new_leaf = nodes[1]

# Verifica quantidade de chaves
assert old_leaf.getNumKeys() == 2
assert new_leaf.getNumKeys() == 3

# Verifica distribuição das chaves e ponteiros
assert get_entries(old_leaf) == [
    (10, 100),
    (20, 200)
]

assert get_entries(new_leaf) == [
    (30, 300),
    (40, 400),
    (50, 500)
]

# Verifica encadeamento das folhas
assert old_leaf.getChildPointer(0) == 1
assert new_leaf.getChildPointer(0) == NO_NEXT

print("Todos os testes passaram!")

Pai: None
Nó encontrado: 0
Chave 20 inserida com sucesso.
Pai: None
Nó encontrado: 0
Chave 10 inserida com sucesso.
Pai: None
Nó encontrado: 0
Chave 40 inserida com sucesso.
Pai: None
Nó encontrado: 0
Chave 30 inserida com sucesso.
Pai: None
Nó encontrado: 0
Chave 50 inserida com sucesso.
Todos os testes passaram!


#### Search post Split test

In [17]:
# Testa buscas nas duas folhas
search_tests = [
    (10, 100),
    (20, 200),
    (30, 300),
    (40, 400),
    (50, 500),
    (60, None) # chave inexistente
]

print("\n--- TESTES DE BUSCA ---")

for key, expected in search_tests:
    result = tree.search(key)

    if result == expected:
        print(f"[OK] Busca pela chave {key}: retornou {result}")
    else:
        print(f"[FALHOU] Busca pela chave {key}: esperado {expected}, retornou {result}")


--- TESTES DE BUSCA ---
[OK] Busca pela chave 10: retornou 100
[OK] Busca pela chave 20: retornou 200
[OK] Busca pela chave 30: retornou 300
[OK] Busca pela chave 40: retornou 400
[OK] Busca pela chave 50: retornou 500
[OK] Busca pela chave 60: retornou None


#### Test with other Split

In [18]:
print("\n--- TESTE DE SEGUNDO SPLIT ---")

tree.insert(60, 600)
tree.insert(70, 700)

# Testa buscas após o segundo split
search_tests = [
    (10, 100),
    (20, 200),
    (30, 300),
    (40, 400),
    (50, 500),
    (60, 600),
    (70, 700),
    (80, None) # Chave inexistente
]

print("\n--- TESTES DE BUSCA APÓS SEGUNDO SPLIT ---")

for key, expected in search_tests:
    result = tree.search(key)

    if result == expected:
        print(f"[OK] Busca pela chave {key}: retornou {result}")
    else:
        print(f"[FALHOU] Busca pela chave {key}: esperado {expected}, retornou {result}")


--- TESTE DE SEGUNDO SPLIT ---
Pai: 0
Nó encontrado: 1
Chave 60 inserida com sucesso.
Pai: 0
Nó encontrado: 1
Chave 70 inserida com sucesso.

--- TESTES DE BUSCA APÓS SEGUNDO SPLIT ---
[OK] Busca pela chave 10: retornou 100
[OK] Busca pela chave 20: retornou 200
[OK] Busca pela chave 30: retornou 300
[OK] Busca pela chave 40: retornou 400
[OK] Busca pela chave 50: retornou 500
[OK] Busca pela chave 60: retornou 600
[OK] Busca pela chave 70: retornou 700
[OK] Busca pela chave 80: retornou None


In [19]:
print("\n--- ESTRUTURA DA ÁRVORE B+ ---")

for node_id, node in tree.nodes.items():
    tipo = "FOLHA" if node.isLeaf() else "INTERNO"
    keys = [node.getKey(i) for i in range(node.getNumKeys())]

    print(f"\nNó {node_id} ({tipo})")
    print(f"  Chaves: {keys}")

    if node.isLeaf():
        print(f"  Próxima folha: {node.getChildPointer(0)}")
        pointers = [node.getPointer(i) for i in range(node.getNumKeys())]
        print(f"  Ponteiros dos registros: {pointers}")
    else:
        children = [node.getChildPointer(i) for i in range(node.getNumKeys() + 1)]
        print(f"  Filhos: {children}")


--- ESTRUTURA DA ÁRVORE B+ ---

Nó 0 (FOLHA)
  Chaves: [10, 20]
  Próxima folha: 1
  Ponteiros dos registros: [100, 200]

Nó 1 (FOLHA)
  Chaves: [30, 40]
  Próxima folha: 3
  Ponteiros dos registros: [300, 400]

Nó 2 (INTERNO)
  Chaves: [30, 50]
  Filhos: [0, 1, 3]

Nó 3 (FOLHA)
  Chaves: [50, 60, 70]
  Próxima folha: 4294967295
  Ponteiros dos registros: [500, 600, 700]
